In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from fundus_data_toolkit.datasets.generic import get_image_dataset
from multistyleseg.experiments.fundus.ensemble import get_ensemble_model
import pandas as pd
from multistyleseg.data.fundus.consts import Lesions, ALL_CLASSES
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from fundus_odmac_toolkit.models.segmentation import batch_segment, postprocess_batch
import cv2
import numpy as np

In [2]:
root_img = Path("/home/clement/Documents/data/IVisionHMR/output/fundus")

dataset = get_image_dataset(root_img, (1536,1536),  precise_autocrop=False)
dataset.return_indices = True

Argument(s) 'always_apply' are not valid for transform MaxSizeTransform
Argument(s) 'always_apply' are not valid for transform PadIfNeeded
Argument(s) 'always_apply' are not valid for transform Normalize


In [3]:
dataloader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=8)

In [4]:
outputs_folders = Path("od_mac")
output_segmentations = outputs_folders / 'segmentation'
output_pickles = outputs_folders / 'pickles'
outputs_folders.mkdir(exist_ok=True, parents=True)
output_segmentations.mkdir(exist_ok=True, parents=True)
output_pickles.mkdir(exist_ok=True, parents=True)

for batch in tqdm(dataloader, total=len(dataloader)):
    indices = batch['index']
    batch_done = False
    for index in indices:
        # Check if output already exists, skip if so
        filepath = Path(dataset.img_filepath['image'][index])
        relative_path = filepath.relative_to(root_img)
        imageId, laterality = relative_path.parts[0], relative_path.parts[1]
        output_pkl = output_pickles / f"{imageId}_{laterality}.pkl"
        if output_pkl.exists():
            batch_done = True
            break
    if batch_done:
        continue
    images = batch['image'].cuda()
    with torch.inference_mode():
        outputs = batch_segment(images, already_normalized=True, use_tta=False, compile=True)
    post_processed_results = postprocess_batch(outputs)
    for i, _ in enumerate(outputs):
        index = batch['index'][i]
        filepath = Path(dataset.img_filepath['image'][index])
        relative_path = filepath.relative_to(root_img)
        imageId, laterality = relative_path.parts[0], relative_path.parts[1]
        
        df = pd.DataFrame(dict(image_id=[imageId], laterality=[laterality],
                               od=[post_processed_results['od_centers'][i]],
                               od_valid=[post_processed_results['od_valid'][i]],
                               macula=[post_processed_results['macula_centers'][i]],
                               macula_valid=[post_processed_results['macula_valid'][i]],))
        output_pkl = output_pickles / f"{imageId}_{laterality}.pkl"
        df.to_pickle(output_pkl)
        predicted_mask = post_processed_results['masks'][i].cpu().numpy().astype(np.uint8)
        output_mask = output_segmentations / f"{imageId}_{laterality}.png"
        cv2.imwrite(str(output_mask), predicted_mask)
        

  0%|          | 0/459 [00:00<?, ?it/s]

Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
